# WebSocket 通信

学习目标：实现有限消息往返，在接受连接前检查演示身份，并验证两条连接在正常结束、提前断开和消息错误后都能清理。

前置知识：async/await、异常处理、HTTP 与 Cookie、依赖注入、JavaScript 事件与浏览器操作。

适用版本：FastAPI 0.141.1、Starlette 1.6、Uvicorn 0.53；真实服务使用 websockets 17.1 的 Sans-I/O 后端。

环境准备：[FastAPI 环境与运行入口](README.md)。

工作目录：content/Web与应用开发/FastAPI。先从空内核运行 Notebook 中的应用内示例，再按末节命令启动端口 8190 的本地服务。浏览器需要启用 JavaScript，实验只发送本地虚构文本。

配套脚本：位于 scripts/19-websocket-communication/。

（1）[app.py](scripts/19-websocket-communication/app.py)：把本篇的身份检查、有限回声和连接计数用于真实服务。

（2）[index.html](scripts/19-websocket-communication/index.html)：用原生 WebSocket API 操作两条连接，显示消息、关闭码和连接数量。

## 1 接受连接，完成一次文字往返

WebSocket 在建立连接后，允许两端沿同一连接持续交换消息。它适合需要双向消息的场景；连接握手与之后的消息往返是不同阶段。

@app.websocket 注册 WebSocket 路由。服务端先 accept，再 receive_text 等待一条文字消息，send_text 返回结果，最后用关闭码 1000 正常结束。客户端使用 websocket_connect 的 with 管理会话。

In [1]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.testclient import TestClient

app = FastAPI()


@app.websocket("/once")
async def echo_once(websocket: WebSocket):
    await websocket.accept()
    text = await websocket.receive_text()
    await websocket.send_text(f"收到：{text}")
    await websocket.close(code=1000)


with TestClient(app) as client:
    with client.websocket_connect("/once") as connection:
        connection.send_text("你好")
        reply = connection.receive_text()
        assert reply == "收到：你好"
        print(reply)  # 预期：收到：你好。

收到：你好


C:\Users\ZHUANG\miniconda3\envs\hands-on-computing\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


## 2 同一连接可以传多条消息

客户端不需要为每条消息重新握手。下面接收两次文字输入，用 send_json 返回结构化结果；JSON 默认通过文字帧传输。选择 receive_text、receive_bytes 或 receive_json 时，应与约定的消息格式对应。

TestClient 的发送和接收是普通同步调用；服务端的 WebSocket 方法需要 await。收到服务端关闭消息后继续读取，会得到 WebSocketDisconnect，可检查其中的关闭码。

In [2]:
@app.websocket("/twice")
async def echo_twice(websocket: WebSocket):
    await websocket.accept()
    # 一次握手建立的连接连续处理两条消息，之后主动正常关闭。
    for number in range(1, 3):
        text = await websocket.receive_text()
        await websocket.send_json({"number": number, "text": text})
    await websocket.close(code=1000)


# 客户端先观察两次回声，再读取关闭事件及其状态码。
with TestClient(app) as client:
    with client.websocket_connect("/twice") as connection:
        for number, text in enumerate(("第一条", "第二条"), start=1):
            connection.send_text(text)
            reply = connection.receive_json()
            assert reply == {"number": number, "text": text}
            print(reply)  # 预期：依次收到 number=1/text=第一条、number=2/text=第二条的字典。
        try:
            connection.receive_text()
        except WebSocketDisconnect as exc:
            assert exc.code == 1000
            print("正常关闭：", exc.code)  # 预期：1000。
        else:
            raise AssertionError("两条消息后应关闭连接")

{'number': 1, 'text': '第一条'}
{'number': 2, 'text': '第二条'}
正常关闭： 1000


## 3 在 accept 之前检查身份和来源

WebSocket 路由也能使用 Depends、Cookie 等依赖。这里为 Alice 和 Bob 生成两份临时 Cookie 值，服务端映射到用户标识；输出不显示这些值。它们只是本地教学身份，不实现账号登录。

浏览器 WebSocket 构造器没有自定义 Authorization 请求头的参数，握手可以携带符合 Cookie 规则的 Cookie。使用 Cookie 时还应检查 Origin：它描述浏览器发起连接的页面来源。来源检查不能替代身份检查，非浏览器客户端也不受浏览器对 Origin 的限制。

本例只接受来自 http://127.0.0.1:8190 的握手。get_demo_user 在路由执行 accept 之前完成两项判断，不通过就抛出 WebSocketException。

In [3]:
from secrets import token_urlsafe
from typing import Annotated

from fastapi import Cookie, Depends, WebSocketException

allowed_origin = "http://127.0.0.1:8190"
session_tokens = {user: token_urlsafe(16) for user in ("alice", "bob")}
session_users = {token: user for user, token in session_tokens.items()}


async def get_demo_user(
    websocket: WebSocket,
    session: Annotated[str | None, Cookie(alias="fastapi_ws_demo")] = None,
) -> str:
    if websocket.headers.get("origin") != allowed_origin:
        raise WebSocketException(code=1008)
    user = session_users.get(session)
    if user is None:
        raise WebSocketException(code=1008)
    return user


DemoUser = Annotated[str, Depends(get_demo_user)]

先用一条很短的受保护路由观察检查成功。TestClient 可以自行设置握手头；这里传入 Origin 只是构造测试输入，不是验证浏览器行为。

In [4]:
@app.websocket("/ws/auth")
async def show_identity(websocket: WebSocket, user: DemoUser):
    await websocket.accept()
    await websocket.send_json({"user": user})
    await websocket.close(code=1000)


with TestClient(app, headers={"Origin": allowed_origin}) as client:
    client.cookies.set("fastapi_ws_demo", session_tokens["alice"])
    with client.websocket_connect("/ws/auth") as connection:
        result = connection.receive_json()
        assert result == {"user": "alice"}
        print("身份检查通过：", result)  # 预期：身份检查通过： {'user': 'alice'}。

身份检查通过：

 {'user': 'alice'}


accept 之前关闭，真实 ASGI 服务会拒绝握手并返回 HTTP 403。此时尚未建立 WebSocket，浏览器不会收到一条关闭码为 1008 的 WebSocket 帧；通常表现为 error 事件和失败的 close 事件。

TestClient 在应用内可以直接观察到关闭消息中的 1008。下面只验证这个应用内结果，浏览器差异在实际页面中观察。

In [5]:
rejected_inputs = (
    ("缺失 Cookie", {}, allowed_origin),
    ("无效 Cookie", {"fastapi_ws_demo": "invalid-demo"}, allowed_origin),
    ("来源不符", {"fastapi_ws_demo": session_tokens["alice"]}, "http://other.test"),
)
for label, cookies, origin in rejected_inputs:
    with TestClient(app, cookies=cookies, headers={"Origin": origin}) as client:
        try:
            with client.websocket_connect("/ws/auth"):
                raise AssertionError("这次握手应被拒绝")
        except WebSocketDisconnect as exc:
            assert exc.code == 1008
            print(label, "应用内拒绝码：", exc.code)  # 预期：缺失 Cookie、无效 Cookie、来源不符的应用内拒绝码均为 1008。

缺失 Cookie 应用内拒绝码： 1008
无效 Cookie 应用内拒绝码： 1008
来源不符 应用内拒绝码： 1008


## 4 用集合管理已接受的连接

connections 保存当前进程中已接受的 WebSocket 对象。接受后加入，finally 中移除，所以客户端提前断开、服务端正常结束和消息错误都经过同一个清理位置。

本例每条连接最多返回三条消息，每条文字按 Python len(text) 计数最多 80。下面用 receive 获取 ASGI 消息，以区分文字、二进制和断开事件；它仍是 WebSocket 的方法，会维护连接状态。

收到断开事件直接退出；收到二进制消息用 1003 拒绝，文字过长用 1009 拒绝。发送期间也可能遇到 WebSocketDisconnect，这时结束处理并清理，不把断开当成仍可继续发送的状态。

In [6]:
connections: set[WebSocket] = set()


@app.websocket("/ws")
async def managed_echo(websocket: WebSocket, user: DemoUser):
    # 身份依赖已完成；这里只将已接受的连接计入集合。
    await websocket.accept()
    connections.add(websocket)
    try:
        for number in range(1, 4):
            message = await websocket.receive()
            # receive 返回协议消息；断开、非文本与超长文本分别处理。
            if message["type"] == "websocket.disconnect":
                return
            if "text" not in message:
                raise WebSocketException(code=1003)
            text = message["text"]
            if len(text) > 80:
                raise WebSocketException(code=1009)
            await websocket.send_json({"user": user, "number": number, "text": text})
        await websocket.close(code=1000)
    except WebSocketDisconnect:
        pass  # 发送时连接已断开；清理由 finally 完成。
    # 正常结束、客户端断开或处理异常，都从集合移除连接。
    finally:
        connections.discard(websocket)

添加一个只返回数量的本地观察接口，避免把连接对象或 Cookie 暴露给客户端。这个集合只属于当前进程；多个 worker 不共享它，不能把一个进程的计数当成整个服务的在线人数。

In [7]:
@app.get("/connections")
async def count_connections() -> dict:
    return {"active": len(connections)}


with TestClient(app) as client:
    result = client.get("/connections")
    assert result.json() == {"active": 0}
    print(result.json())  # 预期：{'active': 0}。

{'active': 0}


## 5 同时连接 Alice 和 Bob

在一个 TestClient 中打开两条 WebSocket。第一条连接握手时识别为 Alice，随后把 Cookie 改为 Bob 再打开第二条；修改 Cookie 不会把已经建立的第一条连接变成 Bob。

这组请求检查两点：回声返回各自识别出的用户，关闭一条连接后只移除对应对象。这里没有跨连接广播。

In [8]:
with TestClient(app, headers={"Origin": allowed_origin}) as client:
    client.cookies.set("fastapi_ws_demo", session_tokens["alice"])
    with client.websocket_connect("/ws") as alice_connection:
        client.cookies.set("fastapi_ws_demo", session_tokens["bob"])
        with client.websocket_connect("/ws") as bob_connection:
            assert client.get("/connections").json() == {"active": 2}
            for user, connection in (("alice", alice_connection), ("bob", bob_connection)):
                connection.send_text("第一条消息")
                reply = connection.receive_json()
                assert reply == {"user": user, "number": 1, "text": "第一条消息"}
                print(reply)  # 预期：alice、bob 各自收到 number=1、text=第一条消息的字典。
            print("两条连接：", client.get("/connections").json())  # 预期：{'active': 2}。
        assert client.get("/connections").json() == {"active": 1}
        print("Bob 断开后：", client.get("/connections").json())  # 预期：{'active': 1}。
    assert client.get("/connections").json() == {"active": 0}
    print("全部断开后：", client.get("/connections").json())  # 预期：{'active': 0}。

{'user': 'alice', 'number': 1, 'text': '第一条消息'}


{'user': 'bob', 'number': 1, 'text': '第一条消息'}
两条连接： {'active': 2}
Bob 断开后： {'active': 1}
全部断开后： {'active': 0}


## 6 观察正常结束和错误关闭

先把三次往返完成，确认服务端用 1000 正常关闭。各次接收都检查内容，避免只数发送次数就宣称完成。

In [9]:
with TestClient(app, headers={"Origin": allowed_origin}) as client:
    client.cookies.set("fastapi_ws_demo", session_tokens["alice"])
    with client.websocket_connect("/ws") as connection:
        for number in range(1, 4):
            connection.send_text(f"消息 {number}")
            reply = connection.receive_json()
            assert reply == {"user": "alice", "number": number, "text": f"消息 {number}"}
            print(reply)  # 预期：user 均为 alice，number 依次为 1、2、3，text 依次为消息 1、消息 2、消息 3。
        try:
            connection.receive_text()
        except WebSocketDisconnect as exc:
            assert exc.code == 1000
            print("三次完成后的关闭码：", exc.code)  # 预期：1000。
        else:
            raise AssertionError("三条消息后应关闭")
    assert client.get("/connections").json() == {"active": 0}

{'user': 'alice', 'number': 1, 'text': '消息 1'}
{'user': 'alice', 'number': 2, 'text': '消息 2'}
{'user': 'alice', 'number': 3, 'text': '消息 3'}
三次完成后的关闭码： 1000


再发送本例不支持的二进制消息和超过限制的文字。它们发生在连接已经建立之后，所以客户端能够接收关闭码。每次错误后再检查连接数量，确认 finally 已执行。

In [10]:
with TestClient(app, headers={"Origin": allowed_origin}) as client:
    client.cookies.set("fastapi_ws_demo", session_tokens["alice"])
    for label, payload, expected in (("二进制", b"demo", 1003), ("过长文字", "x" * 81, 1009)):
        with client.websocket_connect("/ws") as connection:
            if isinstance(payload, bytes):
                connection.send_bytes(payload)
            else:
                connection.send_text(payload)
            try:
                connection.receive_text()
            except WebSocketDisconnect as exc:
                assert exc.code == expected
                print(label, "关闭码：", exc.code)  # 预期：二进制关闭码为 1003，过长文字为 1009。
            else:
                raise AssertionError("错误消息应导致连接关闭")
        assert client.get("/connections").json() == {"active": 0}
    print("错误连接清理后：", client.get("/connections").json())  # 预期：{'active': 0}。

二进制 关闭码： 1003
过长文字 关闭码： 1009
错误连接清理后： {'active': 0}


## 7 浏览器端按连接状态发送和关闭

配套页面用原生 WebSocket API，分别保存连接 A 和连接 B。open 事件表示可以发送；message 事件中的 data 是服务端发来的消息；close 事件提供关闭码。发送前检查 readyState 等于 WebSocket.OPEN，结束时调用 close。

页面的核心调用如下，完整按钮行为位于 index.html。先通过本地演示接口取得 HttpOnly Cookie，再建立连接；凭据不放进 URL 或页面输出。

```javascript
await fetch('/demo-session/alice', {method: 'POST'});
const socket = new WebSocket('ws://127.0.0.1:8190/ws');
socket.addEventListener('open', () => socket.send('你好'));
socket.addEventListener('message', (event) => {
  // 普通文本通过 textContent 显示，不当作 HTML 执行。
  const item = document.createElement('li');
  // 预期：新增一条回声，user 为 alice，number 为 1，text 为你好。
  item.textContent = event.data;
  document.querySelector('#messages').append(item);
});
socket.addEventListener('close', (event) => {
  // 预期：正常关闭为 1000；本段只发送一次，连接会保持到主动关闭或其他关闭条件发生。
  document.querySelector('#status').textContent = `关闭码：${event.code}`;
});
```

演示接口只允许选择 Alice 或 Bob，目的是建立本地 Cookie 输入；这不是正式登录接口。身份在握手时检查，本例没有令牌到期或撤销机制；长期连接中的业务授权需要另按消息和资源检查。

## 8 启动服务并操作两条连接

配套 app.py 保存本篇管理连接的实现，另外用 FileResponse 提供页面，并用演示接口设置或清除 Cookie。路径根据 app.py 自身位置定位，不依赖启动终端之外的章节文件。

Step 1：在课程工作目录的独立终端启动单进程服务。

```powershell
python -m uvicorn app:app --app-dir scripts/19-websocket-communication --host 127.0.0.1 --port 8190 --workers 1 --ws websockets-sansio --no-access-log
```

Step 2：在浏览器打开 http://127.0.0.1:8190。

（1）先点“无身份连接测试”。页面显示连接失败，活动连接数保持 0。真实服务拒绝握手时返回 HTTP 403；浏览器 close 事件的失败码可能显示为 1006，不能据此把它当成服务端发送的关闭帧。

（2）依次点“连接 A”和“连接 B”，等两个连接都显示打开，再点“刷新连接数”，应为 2。分别点“发送 A”和“发送 B”，观察 Alice 与 Bob 的第一条回声。

（3）点“断开 A”，刷新后连接数应为 1。继续给 B 发送到第三条消息，B 收到第三条回声后以 1000 关闭，刷新后连接数为 0。

（4）重新连接 A，点“发送超长消息 A”，观察 1009 关闭和连接数恢复为 0。

消息只在页面中显示，不保存到数据库。页面使用 textContent 输出文字；关闭按钮和离开页面时的 pagehide 处理都会请求关闭连接。断开后的数量以服务端 /connections 返回值为准。

![浏览器中 Alice 和 Bob 各收到第一条回声，活动连接数为 2](image/19-two-connections.png)

图中是本例两个连接同时打开、分别收到第一条回声的页面。

## 9 关闭连接、服务与演示身份

Step 1：在页面断开 A 和 B，刷新连接数，确认显示 0。

Step 2：点“清除演示身份”，再关闭本实验的浏览器页面。

Step 3：在服务终端按 Ctrl+C，等待 Uvicorn 输出应用关闭完成并返回提示符。

只关闭页面不会停止 Uvicorn；只删除 Cookie 也不会自动撤销已建立连接。本例已保存的用户标识和连接集合属于当前进程，服务结束后不再保留。下面清空 Notebook 自己的演示映射；它与独立终端中的服务是两份独立输入。

In [11]:
assert not connections
session_tokens.clear()
session_users.clear()
print("Notebook 活动连接数：", len(connections))  # 预期：Notebook 活动连接数： 0。
# 独立 Uvicorn 服务仍需按上面的终端步骤停止。

Notebook 活动连接数： 0


## 本章小结

（1）先接受连接，再按约定格式收发消息；一条连接可以完成多次往返。

（2）身份与来源检查放在 accept 前；握手拒绝与连接建立后的关闭码是两种不同结果。

（3）连接成功后加入集合，finally 中移除；正常关闭、提前断开和输入错误都要检查清理。

（4）浏览器等 open 后再发送，按 close 更新界面；连接集合只在当前进程有效。

## 练习

（1）把每条连接的消息上限改为两条。验证标准：两条回声的 number 为 1 和 2，随后关闭码为 1000，活动连接数恢复 0。

（2）保留正确 Cookie，只改变 Origin。验证标准：应用内握手被拒绝，connections 不增加；说明为什么单有 Cookie 不能省略浏览器来源检查。

（3）打开两条连接，让 A 发送过长消息。验证标准：A 以 1009 关闭，B 仍能正常往返，连接数先从 2 变为 1，再在 B 断开后变为 0。

（4）在浏览器发送文字 &lt;b&gt;hello&lt;/b&gt;。验证标准：页面按普通文字展示它，不能把消息内容解释成加粗的 HTML；随后关闭连接和服务。

提示：Notebook 已执行清理时，从空内核重新运行相关示例；修改浏览器服务后重启它。观察状态码或关闭码之外，还要观察剩余连接能否继续工作和最终计数。

## 参考与引用来源

- **FastAPI 官方文档**：[WebSockets](https://fastapi.tiangolo.com/advanced/websockets/) 的 Create a websocket、Await for messages、Using Depends and others、Handling disconnections and multiple clients，用于路由、身份依赖、异常处理与单进程连接集合；[WebSocket.send 源码](https://fastapi.tiangolo.com/reference/websockets/#fastapi.WebSocket.send)，用于发送遇到连接错误时的 WebSocketDisconnect；[Response Cookies](https://fastapi.tiangolo.com/advanced/response-cookies/)，用于本地演示接口设置 Cookie。
- **Starlette 官方文档**：[WebSockets](https://starlette.dev/websockets/) 的 Accepting、Sending/Receiving data、Closing、Sending and receiving messages、Send Denial Response，用于消息格式、状态维护、accept 前的 HTTP 403 与关闭；[TestClient](https://starlette.dev/testclient/#testing-websocket-sessions)，用于会话上下文管理器及同步测试调用；[Responses](https://starlette.dev/responses/#set-cookie)，用于 Cookie 属性、删除与 FileResponse。
- **WHATWG WebSockets 标准**：[WebSocket 接口](https://websockets.spec.whatwg.org/#the-websocket-interface)、[建立连接](https://websockets.spec.whatwg.org/#connections) 与 [事件反馈](https://websockets.spec.whatwg.org/#feedback-from-the-protocol)，用于构造参数、Cookie、readyState、消息事件和握手失败时浏览器隐藏细节的行为。
- **OWASP Cheat Sheet Series**：[WebSocket Security Cheat Sheet](https://cheatsheetseries.owasp.org/cheatsheets/WebSocket_Security_Cheat_Sheet.html) 的 Origin Header Validation、SameSite Cookies、Message-Level Authorization 与 Input Validation，用于握手来源、Cookie 和消息边界；[DOM based XSS Prevention](https://cheatsheetseries.owasp.org/cheatsheets/DOM_based_XSS_Prevention_Cheat_Sheet.html)，用于通过 textContent 显示不可信文字。
- **RFC Editor**：[RFC 6455 第 7.4.1 节](https://www.rfc-editor.org/rfc/rfc6455.html#section-7.4.1)，用于关闭码 1000、1003、1006、1008、1009 的含义。
- **Uvicorn 官方文档**：[Settings](https://uvicorn.dev/settings/#implementation) 的 Implementation、Logging，用于 websockets-sansio 后端和本地启动参数。
- **Python 官方文档**：[secrets.token_urlsafe](https://docs.python.org/3/library/secrets.html#secrets.token_urlsafe)，用于临时 Cookie 输入；[Compound statements 的 finally](https://docs.python.org/3/reference/compound_stmts.html#finally-clause)，用于统一连接清理。
- **MDN 浏览器文档**：[Writing WebSocket client applications](https://developer.mozilla.org/en-US/docs/Web/API/WebSockets_API/Writing_WebSocket_client_applications) 的 Handling disconnect 与 Working with the bfcache，用于事件处理和 pagehide 时关闭连接；[AbortSignal.timeout](https://developer.mozilla.org/en-US/docs/Web/API/AbortSignal/timeout_static)，用于页面 HTTP 准备与计数请求的 3000 毫秒等待上限。